## 埋め込み

前回の例では、長さ`vocab_size`の高次元の単語袋ベクトルを操作し、低次元の位置表現ベクトルから疎なワンホット表現に明示的に変換していました。このワンホット表現はメモリ効率が悪く、さらに各単語は互いに独立して扱われているため、ワンホットエンコードされたベクトルは単語間の意味的類似性を表現しません。

このユニットでは、<strong>News AG</strong>データセットの探索を続けます。まず、データを読み込み、前のノートブックからいくつかの定義を取得しましょう。


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## 埋め込みとは？

<strong>埋め込み</strong>の考え方は、単語をその意味をなんらかの形で反映する低次元の密なベクトルで表現することです。後ほど意味のある単語埋め込みを作成する方法について説明しますが、今は埋め込みを単語ベクトルの次元数を減らす方法と考えましょう。

つまり、埋め込み層は単語を入力として受け取り、指定した`embedding_size`の出力ベクトルを生成します。ある意味で`Linear`層に非常に似ていますが、ワンホットエンコードされたベクトルを取るのではなく、単語番号を入力として受け取ることができます。

ネットワークの最初の層として埋め込み層を使用することで、袋詰め単語モデルから<strong>埋め込み袋</strong>モデルに切り替えることができます。ここでは、テキスト内の各単語を対応する埋め込みに変換し、それらすべての埋め込みに対して`sum`、`average`、`max`のような集計関数を計算します。

![5つのシーケンス単語に対する埋め込み分類器の図](../../../../../translated_images/ja/embedding-classifier-example.b77f021a7ee67eee.webp)

私たちの分類ニューラルネットワークは、埋め込み層の後に集計層、その上に線形分類器を配置して開始します:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### 可変長シーケンスへの対応

このアーキテクチャの結果として、ネットワークへのミニバッチは特定の方法で作成する必要があります。前のユニットでは、bag-of-wordsを使うとき、ミニバッチ内のすべてのBoWテンソルは実際のテキストシーケンスの長さに関わらず等しいサイズ`vocab_size`を持っていました。単語埋め込みに移行すると、各テキストサンプル内の単語数が変動し、これらのサンプルをミニバッチにまとめる際にパディングを適用する必要が出てきます。

これは、データソースに`collate_fn`関数を提供する同じ手法で行うことができます：


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### 埋め込み分類器のトレーニング

適切なデータローダーが定義できたので、前のユニットで定義したトレーニング関数を使ってモデルをトレーニングできます。


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> <strong>注意</strong>: 時間節約のため、ここでは25,000件のレコード（完全な1エポック未満）だけトレーニングしていますが、トレーニングを続けたり、複数エポック学習する関数を書いたり、学習率のパラメーターを調整してより高い精度を目指すことができます。約90%の精度を達成できるはずです。


### EmbeddingBag レイヤーと可変長シーケンスの表現

以前のアーキテクチャでは、ミニバッチに収めるためにすべてのシーケンスを同じ長さにパディングする必要がありました。これは可変長のシーケンスを表現する最も効率的な方法ではありません。別のアプローチとして、すべてのシーケンスのオフセットを1つの大きなベクトルに格納する **offset** ベクトルを使用する方法があります。

![オフセットシーケンス表現を示す画像](../../../../../translated_images/ja/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Note**: 上の図では文字のシーケンスを示していますが、私たちの例では単語のシーケンスを扱っています。ただし、オフセットベクトルでシーケンスを表現する一般原則は同じです。

オフセット表現を扱うために、[`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) レイヤーを使います。これは `Embedding` に似ていますが、コンテンツベクトルとオフセットベクトルを入力として受け取り、また `mean`、`sum`、もしくは `max` となり得る平均化レイヤーを含みます。

以下は `EmbeddingBag` を使った修正済みのネットワークです：


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

データセットをトレーニング用に準備するには、オフセットベクターを準備する変換関数を提供する必要があります：


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

注意すべきは、これまでのすべての例とは異なり、今回のネットワークは異なるサイズの2つのパラメーター、データベクターとオフセットベクターを受け取るということです。同様に、データローダーも2つではなく3つの値を提供します。テキストとオフセットの両方のベクターが特徴量として提供されます。したがって、それに対応するためにトレーニング関数を少し調整する必要があります。


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## セマンティック埋め込み：Word2Vec

前の例では、モデルの埋め込み層が単語をベクトル表現にマップすることを学習しましたが、この表現はあまり意味的な意味を持っていませんでした。類似した単語や同義語が、あるベクトル距離（例：ユークリッド距離）において互いに近いベクトルに対応するようなベクトル表現を学習できると良いでしょう。

それには、特定の方法で大量のテキストコレクションに対して埋め込みモデルを事前学習する必要があります。セマンティック埋め込みを学習する最初の方法の一つが[Word2Vec](https://en.wikipedia.org/wiki/Word2vec)です。これは単語の分散表現を生成するために使われる2つの主なアーキテクチャに基づいています：

 - **Continuous bag-of-words（CBoW）** — このアーキテクチャでは、周囲の文脈から単語を予測するようにモデルを学習します。n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$が与えられたとき、モデルの目標は$(W_{-2},W_{-1},W_1,W_2)$から$W_0$を予測することです。
 - **Continuous skip-gram** はCBoWの逆です。モデルは現在の単語を予測するために周囲の文脈単語を使用します。

CBoWは高速で、skip-gramは遅いですが、稀な単語を表現するのに優れています。

![単語をベクトルに変換するCBoWとSkip-Gramアルゴリズムの両方を示した画像](../../../../../translated_images/ja/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Google Newsデータセットで事前学習されたword2vec埋め込みを試すには、<strong>gensim</strong>ライブラリを使えます。以下に「neural」に最も類似した単語を示します。

> **注意：** 単語ベクトルを初めて作成するときは、ダウンロードに時間がかかることがあります！


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


単語からベクトル埋め込みを計算し、分類モデルのトレーニングに使用することもできます（分かりやすさのためにベクトルの最初の20成分のみを表示しています）：


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

セマンティカル埋め込みの素晴らしい点は、ベクトル表現を操作してセマンティクスを変えることができることです。例えば、ベクトル表現が単語 *king* と *woman* にできるだけ近く、単語 *man* からはできるだけ遠い単語を見つけるように求めることができます：


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW と Skip-Grams はどちらも「予測的」埋め込みであり、局所的な文脈のみを考慮します。Word2Vec はグローバルな文脈を活用しません。 

**FastText** は Word2Vec を基にしており、各単語とその単語内の文字 n-グラムに対するベクトル表現を学習します。これらの表現の値は各トレーニングステップで平均化されて一つのベクトルになります。これにより事前学習の計算は大幅に増えますが、単語埋め込みがサブワード情報を符号化できるようになります。 

もう一つの方法である **GloVe** は共起行列のアイデアを利用し、ニューラル手法を用いて共起行列をより表現力があり非線形な単語ベクトルに分解します。

gensim は複数の異なる単語埋め込みモデルをサポートしているため、埋め込みを FastText や GloVe に変えて例を試すことができます。


## PyTorchで事前学習済みの埋め込みを使用する

上記の例を修正して、埋め込み層の行列をWord2Vecなどの意味的な埋め込みで事前に埋めることができます。事前学習済みの埋め込みと私たちのテキストコーパスの語彙が一致しない可能性が高いことを考慮する必要があるため、欠けている単語の重みはランダムな値で初期化します：


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


それではモデルの訓練を始めましょう。埋め込み層のサイズが大きく、パラメーター数が大幅に増加しているため、訓練にかかる時間は前の例よりもかなり長くなっています。また、そのため、過学習を避けたい場合には、より多くのサンプルでモデルを訓練する必要があるかもしれません。


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

私たちの場合、大幅な精度向上は見られません。これは語彙が大きく異なることが原因と思われます。 
異なる語彙の問題を克服するために、以下のいずれかの解決策を使うことができます:
* 自分の語彙でword2vecモデルを再訓練する
* 事前学習済みword2vecモデルの語彙を使ってデータセットを読み込む。読み込み時に使用する語彙を指定できます。

後者の方法のほうが簡単に思えます。特にPyTorchの`torchtext`フレームワークは組み込みの埋め込みサポートを含んでいるためです。例えば、以下のようにGloVeベースの語彙を生成できます:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Loaded vocabulary has the following basic operations:
* `vocab.stoi` dictionary allows us to convert word into its dictionary index
* `vocab.itos` does the opposite - converts number into word
* `vocab.vectors` is the array of embedding vectors, so to get the embedding of a word `s` we need to use `vocab.vectors[vocab.stoi[s]]`

Here is the example of manipulating embeddings to demonstrate the equation **kind-man+woman = queen** (I had to tweak the coefficient a bit to make it work):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

これらの埋め込みを使って分類器を訓練するには、まずGloVeの語彙を使ってデータセットをエンコードする必要があります：


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

上記で見たように、すべてのベクトル埋め込みは `vocab.vectors` 行列に格納されています。これにより、埋め込み層の重みへ単純なコピーでこれらの重みを読み込むのが非常に簡単になります。


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

では、モデルを訓練して結果が良くなるか見てみましょう：


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

精度が大幅に向上していない理由の一つは、データセットの中のいくつかの単語が事前学習済みのGloVe語彙に存在せず、実質的に無視されているためです。この問題を克服するために、私たちは自身のデータセット上で独自の埋め込みを訓練することができます。 


## コンテキスト埋め込み

Word2Vec のような従来の事前学習済み埋め込み表現の主な制限の 1 つは、語義曖昧性の問題です。事前学習済みの埋め込みは文脈中の単語の意味の一部を捉えることができますが、単語のあらゆる意味は同じ埋め込みに符号化されます。これは、使用される文脈によって異なる意味を持つ『play』のような多くの単語において、下流のモデルで問題を引き起こす可能性があります。

例えば、『play』という単語は次の二つの文ではかなり異なる意味を持っています。
- 私は劇場で<strong>演劇</strong>を観に行きました。
- ジョンは友達と<strong>遊びたい</strong>と思っています。

上記の事前学習済み埋め込みでは、この『play』の両方の意味を同じ埋め込みで表現しています。この制限を克服するために、言語モデルに基づいた埋め込みを構築する必要があります。言語モデルは大規模なテキストコーパスで訓練され、単語が異なる文脈でどのように組み合わされるかを<em>理解</em>しています。コンテキスト埋め込みの議論はこのチュートリアルの範囲外ですが、次のユニットで言語モデルについて説明するときに再び取り上げます。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
